# Extraction des timeseries individuelles — OEDI
Téléchargement des séries temporelles par bâtiment depuis le data lake public OEDI (ResStock 2025, AMY2018).  
Chaque fichier = 1 bâtiment × 35 040 pas de 15 min (année 2018) × 192 colonnes.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT           = Path().resolve().parent.parent
DATA_PROCESSED = ROOT / 'data' / 'processed'
DATA_RAW       = ROOT / 'data' / 'raw'

OEDI_BASE = (
    'https://oedi-data-lake.s3.amazonaws.com/'
    'nrel-pds-building-stock/end-use-load-profiles-for-us-building-stock/'
    '2025/resstock_amy2018_release_1/'
    'timeseries_individual_buildings/by_state/upgrade=0'
)

## 1. Filtrer les bâtiments dans les métadonnées
Critères : maison individuelle plein pied, chauffage électrique, Central AC,  
sans VE / piscine / panneaux solaires, zones climatiques tempérées (3A, 4A, 5A).

In [ ]:
raw = pd.read_parquet(DATA_RAW / 'upgrade0.parquet', columns=[
    'bldg_id',
    'in.state',
    'in.geometry_building_type_recs',
    'in.geometry_stories',
    'in.electric_vehicle_ownership',
    'in.misc_pool',
    'in.has_pv',
    'in.hvac_cooling_type',
    'in.hvac_heating_type',
    'in.heating_fuel',
    'in.ashrae_iecc_climate_zone_2004',
])

mask = (
    (raw['in.geometry_building_type_recs'] == 'Single-Family Detached') &
    (raw['in.geometry_stories'] == '1') &
    (raw['in.electric_vehicle_ownership'] == 'No') &
    (raw['in.misc_pool'] == 'None') &
    (raw['in.has_pv'] == 'No') &
    (raw['in.hvac_cooling_type'] == 'Central AC') &
    (raw['in.hvac_heating_type'] == 'Ducted Heating') &
    (raw['in.heating_fuel'] == 'Electricity') &
    (raw['in.ashrae_iecc_climate_zone_2004'].isin(['3A', '4A', '5A']))
)

candidates = raw[mask][['bldg_id', 'in.state', 'in.ashrae_iecc_climate_zone_2004']].reset_index(drop=True)
print(f'{len(candidates):,} bâtiments correspondent aux critères')
candidates.head(10)

## 2. Télécharger la timeserie d'un bâtiment
Les fichiers sont publics sur OEDI — aucune clé AWS requise.  
URL : `{OEDI_BASE}/state={state}/{bldg_id}-0.parquet`

In [ ]:
def download_timeseries(bldg_id: int, state: str, save: bool = True) -> pd.DataFrame:
    url = f'{OEDI_BASE}/state={state}/{bldg_id}-0.parquet'
    print(f'Téléchargement : {url}')
    df = pd.read_parquet(url)
    if save:
        out = DATA_RAW  / f'{bldg_id}-0.parquet'
        df.to_parquet(out)
        print(f'Sauvegardé : {out}')
    print(f'Shape : {df.shape} | {df["timestamp"].iloc[0]} → {df["timestamp"].iloc[-1]}')
    return df

In [ ]:
182255 in candidates["bldg_id"].values

In [ ]:
# Exemple : bldg_id 347201 — Texas, zone 3A
bldg_id = 347201
state   = candidates.loc[candidates['bldg_id'] == bldg_id, 'in.state'].values[0]

df_ts = download_timeseries(bldg_id, state)
df_ts.head()



In [ ]:
bldg_id = 347201
state = "VA"

df_ts = download_timeseries(bldg_id, state)

## 3. Télécharger plusieurs bâtiments en batch

In [ ]:
# Modifier N pour changer le nombre de bâtiments à télécharger
N = 5
sample = candidates.head(N)

ts_dict = {}
for _, row in sample.iterrows():
    bid, state = int(row['bldg_id']), row['in.state']
    try:
        ts_dict[bid] = download_timeseries(bid, state, save=True)
    except Exception as e:
        print(f'  Erreur bldg_id={bid} : {e}')

print(f'\n{len(ts_dict)} bâtiments téléchargés')